# 01 - GEE Observation Extraction

## Purpose
Extract monthly Google Earth Engine observations for manually supplied Sagil grid-cell centroids.

## Inputs
- `config/project_config.yaml`
- `data/input/coordinates.csv`
- Google Earth Engine credentials from `GEE_PROJECT_ID` or ignored local credentials

## Outputs
- `data/raw/monthly/<sample_id>.csv`
- `data/metadata/sample_index.csv`
- `data/metadata/extraction_manifest.json`

## Notes
Each coordinate represents the centroid of a 1 km x 1 km square cell. Monthly rows use full calendar-month intervals `[month_start, next_month_start)`.

## 1. Configure Extraction Paths
Defines the project root, config path, and explicit switches for slow GEE extraction.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "config" / "project_config.yaml"
RUN_GEE_EXTRACTION = True
FORCE_REFETCH = False
SAMPLE_LIMIT = None

## 2. Load Config and Validate Coordinates
Checks coordinate schema, unique IDs/grid cells, monthly schedule length, and 1 km square-cell geometry before any Earth Engine call.

In [2]:
from src.validation import (
    build_month_schedule,
    build_square_cell,
    load_coordinates,
    load_project_config,
    path_from_config,
    validate_cell_area,
)

config = load_project_config(CONFIG_PATH)
coordinates = load_coordinates(path_from_config(config, "paths", "coordinates_csv"))
months = build_month_schedule(config["gee"]["first_month"], config["gee"]["last_month"])

print(f"Coordinates: {len(coordinates)}")
print(f"Months: {len(months)} ({months[0]} to {months[-1]})")

grid_config = config["grid"]
for _, row in coordinates.head(5).iterrows():
    cell = build_square_cell(
        row["latitude"],
        row["longitude"],
        grid_config["half_width_m"],
        grid_config["projected_crs"],
    )
    validate_cell_area(
        cell["cell_area_m2"],
        grid_config["expected_area_m2"],
        grid_config["area_tolerance_fraction"],
    )

coordinates.head()

Coordinates: 49
Months: 66 (2021-01-01 to 2026-06-01)


,sample_id,grid_row,grid_col,latitude,longitude,study_area,cell_size_m
0,sagil_1,0,0,2.341985,102.629998,"Sagil, Tangkak, Johor",1000
1,sagil_2,0,1,2.342001,102.638985,"Sagil, Tangkak, Johor",1000
2,sagil_3,0,2,2.342016,102.647971,"Sagil, Tangkak, Johor",1000
3,sagil_4,0,3,2.342031,102.656958,"Sagil, Tangkak, Johor",1000
4,sagil_5,0,4,2.342046,102.665945,"Sagil, Tangkak, Johor",1000


## 3. Run Monthly GEE Extraction
Calls Earth Engine only when `RUN_GEE_EXTRACTION = True`. This writes one raw monthly CSV per coordinate and updates metadata files.

In [3]:
from src.gee_monthly_extraction import MonthlyExtractionConfig, extract_monthly_observations

if RUN_GEE_EXTRACTION:
    extraction_config = MonthlyExtractionConfig(
        config_path=str(CONFIG_PATH),
        force=FORCE_REFETCH,
        sample_limit=SAMPLE_LIMIT,
        verbose=True,
    )
    sample_index = extract_monthly_observations(extraction_config)
    print(sample_index.shape)
    sample_index.head()
else:
    print("RUN_GEE_EXTRACTION is False. Set it to True only when you want to call Earth Engine.")

GEE cells:   0%|          | 0/49 [00:00<?, ?it/s]

(49, 11)


## 4. Preview Sample Index
Loads the generated sample index when available to verify raw CSV lookup paths.

In [4]:
import pandas as pd

sample_index_path = PROJECT_ROOT / config["paths"]["sample_index_csv"]
if sample_index_path.exists():
    sample_index = pd.read_csv(sample_index_path)
    print(sample_index.shape)
    display(sample_index.head())
else:
    print(f"Sample index not found yet: {config['paths']['sample_index_csv']}")

(49, 11)


,sample_id,grid_row,grid_col,latitude,longitude,cell_area_m2,first_month,last_month,row_count,raw_csv,extraction_status
0,sagil_1,0,0,2.341985,102.629998,1000000.0,2021-01-01,2026-06-01,66,data\raw\monthly\sagil_1.csv,ok
1,sagil_2,0,1,2.342001,102.638985,1000000.0,2021-01-01,2026-06-01,66,data\raw\monthly\sagil_2.csv,ok
2,sagil_3,0,2,2.342016,102.647971,1000000.0,2021-01-01,2026-06-01,66,data\raw\monthly\sagil_3.csv,ok
3,sagil_4,0,3,2.342031,102.656958,1000000.0,2021-01-01,2026-06-01,66,data\raw\monthly\sagil_4.csv,ok
4,sagil_5,0,4,2.342046,102.665945,1000000.0,2021-01-01,2026-06-01,66,data\raw\monthly\sagil_5.csv,ok
